# Spateo Alignment Parameter Tuning for MerXen

Interactive notebook for testing Spateo alignment settings on a MerXen Xenium/MERSCOPE pair without rewriting full SpatialData zarrs. The goal is to tune rigid, partial-overlap, and non-rigid behavior with fast centroid overlays before committing a configuration to the Nextflow pipeline.

This notebook follows the Spateo alignment tutorials' intended pattern:

- build two AnnData slices with `obsm["spatial"]` coordinates
- optionally preprocess and select features
- compute PCA jointly across both slices and align with `rep_layer="X_pca"`, `rep_field="obsm"`, `dissimilarity="cos"`
- compare `align_spatial`, `align_spatial_rigid`, and `align_spatial_nonrigid`
- tune partial-overlap and non-rigid parameters before applying transforms to full data

Nothing here writes aligned zarrs unless you explicitly add that yourself.

## Documentation Notes

Spateo docs checked while creating this notebook:

- [Alignment tutorial index](https://spateo-release.readthedocs.io/en/latest/tutorials/notebooks/3_alignment.html)
- [Basic usage of Spateo alignment for 2D slices](https://spateo-release.readthedocs.io/en/latest/tutorials/notebooks/3_alignment/1.%20Basic%20usage%20of%20Spateo%20alignment%20for%202D%20slices.html)
- [Nonrigid alignment of Spateo alignment for 2D slices](https://spateo-release.readthedocs.io/en/latest/tutorials/notebooks/3_alignment/2.%20Nonrigid%20alignment%20of%20Spateo%20alignment%20for%202D%20slices.html)
- [Partial alignment of Spateo for 2D slices](https://spateo-release.readthedocs.io/en/latest/tutorials/notebooks/3_alignment/3.%20Partial%20alignment%20of%20Spateo%20for%202D%20slices.html)
- [Improve efficiency and scalibity](https://spateo-release.readthedocs.io/en/latest/tutorials/notebooks/3_alignment/4.%20Improve%20efficiency%20and%20scalibity.html)

Implementation takeaways:

- Basic 2D usage recommends standard preprocessing, HVG selection, and joint PCA across both slices before calling `st.align.morpho_align` with PCA features.
- Non-rigid 2D alignment identifies `beta`, `lambdaVF`, and `K` as the main flexibility controls. Increasing `beta` or `K`, or decreasing `lambdaVF`, makes the deformation more flexible.
- Partial alignment recommends increasing `partial_robust_level` for partial-overlap cases and warns that overly flexible non-rigid settings can distort non-overlapping tissue.
- Efficiency/scalability recommends GPU, SVI, sparse calculation, chunking, and downsampling for large datasets. It also shows a hard partial-overlap case where SVI fails but non-SVI succeeds, so this notebook supports testing SVI on subsampled data.

For MerXen P7513, rigid alignment currently looks useful while the full non-rigid result can collapse inward. Start with conservative non-rigid settings and judge the rigid panel separately from the non-rigid panel.

In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import anndata as ad
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import torch
from scipy.spatial import cKDTree

from merxen.alignment.features import (
    _robust_centroid_xy,
    build_alignment_adata,
    prepare_spateo_features,
    shared_gene_subset,
)
from merxen.alignment.register import _apply_spateo_import_shims, _resolve_device

warnings.filterwarnings("ignore")
_apply_spateo_import_shims()
import spateo as st

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 250

# Spateo 1.1.1 expects a CUDA_VISIBLE_DEVICES-style GPU id such as "0".
DEVICE = _resolve_device("auto")
print("torch cuda available:", torch.cuda.is_available())
print("Spateo device argument:", DEVICE)
print("Spateo version:", st.__version__)

## Paths

Set `PAIR_ID` to any processed sample pair. P5011 is the currently interesting failure case, while P7513 is a useful known-good comparison. `EXPERIMENT_ROOT` stores coordinate CSVs, metadata JSON, and overlay PNGs only.


In [ ]:
AVAILABLE_PAIR_IDS = ["P1212", "P5011", "P7113", "P7513"]
PAIR_ID = "P5011"  # Try "P7513" as the known-good reference from earlier tuning.

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "MerXen":
    REPO_ROOT = cwd.parent
elif cwd.name == "MerXen":
    REPO_ROOT = cwd
else:
    REPO_ROOT = Path("/home/becalia/code/MerXen")

if PAIR_ID not in AVAILABLE_PAIR_IDS:
    raise ValueError(f"PAIR_ID must be one of {AVAILABLE_PAIR_IDS}, got {PAIR_ID!r}")

MERSCOPE_ZARR = REPO_ROOT / "results" / PAIR_ID / "merscope" / "latest" / "latest_spatialdata.zarr"
XENIUM_ZARR = REPO_ROOT / "results" / PAIR_ID / "xenium" / "latest" / "latest_spatialdata.zarr"
PIPELINE_ALIGNMENT_DIR = REPO_ROOT / "results" / PAIR_ID / "alignment" / "align_out"
PIPELINE_QC_DIR = REPO_ROOT / "results" / PAIR_ID / "alignment_qc" / "alignment_qc_out"
EXPERIMENT_ROOT = REPO_ROOT / "alignment_test" / f"{PAIR_ID}_spateo_param_tuning"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

for label, path in {
    "MERSCOPE_ZARR": MERSCOPE_ZARR,
    "XENIUM_ZARR": XENIUM_ZARR,
    "PIPELINE_ALIGNMENT_DIR": PIPELINE_ALIGNMENT_DIR,
    "PIPELINE_QC_DIR": PIPELINE_QC_DIR,
    "EXPERIMENT_ROOT": EXPERIMENT_ROOT,
}.items():
    print(f"{label}: {path.resolve()}  exists={path.exists()}")


## Load MerXen Data as AnnData

Xenium is fixed/reference, MERSCOPE is moving. This uses MerXen's alignment feature builder so the inputs match the pipeline.

In [ ]:
xenium_sdata = sd.read_zarr(XENIUM_ZARR)
merscope_sdata = sd.read_zarr(MERSCOPE_ZARR)

fixed_base = build_alignment_adata(xenium_sdata, platform="XENIUM")
moving_base = build_alignment_adata(merscope_sdata, platform="MERSCOPE")

print("fixed Xenium:", fixed_base.shape)
print("moving MERSCOPE:", moving_base.shape)
print("raw shared genes:", len(set(fixed_base.var_names) & set(moving_base.var_names)))

## Pipeline QC Snapshot and Geometry Sanity Check

This cell compares the saved pipeline QC result for the selected pair and checks whether the aligned MERSCOPE shape layer has pathological centroids. P5011 previously had a single invalid warped polygon whose Shapely centroid landed millions of coordinate units away from its own tiny bounds; that stretched the overlay and grid QC.


In [ ]:
qc_metrics_path = PIPELINE_QC_DIR / f"{PAIR_ID}_alignment_qc_metrics.csv"
if qc_metrics_path.exists():
    display(pd.read_csv(qc_metrics_path))
else:
    print(f"No saved pipeline QC metrics at {qc_metrics_path}")

moving_shape_key = moving_base.uns["merxen_alignment"]["shape_key"]
aligned_shape_path = MERSCOPE_ZARR / "shapes" / moving_shape_key / "shapes.parquet"
print("MERSCOPE shape selected by build_alignment_adata:", moving_shape_key)

if aligned_shape_path.exists():
    gdf = gpd.read_parquet(aligned_shape_path)
    raw_cent = gdf.geometry.centroid
    bounds = gdf.geometry.bounds
    bad_centroid = (
        ~np.isfinite(raw_cent.x)
        | ~np.isfinite(raw_cent.y)
        | (raw_cent.x < bounds.minx)
        | (raw_cent.x > bounds.maxx)
        | (raw_cent.y < bounds.miny)
        | (raw_cent.y > bounds.maxy)
    )
    robust_x, robust_y = _robust_centroid_xy(gdf)
    print("rows:", len(gdf))
    print("invalid geometries:", int((~gdf.geometry.is_valid).sum()))
    print("pathological centroids:", int(bad_centroid.sum()))
    print("raw centroid x/y range:", (float(raw_cent.x.min()), float(raw_cent.x.max())), (float(raw_cent.y.min()), float(raw_cent.y.max())))
    print("robust centroid x/y range:", (float(np.nanmin(robust_x)), float(np.nanmax(robust_x))), (float(np.nanmin(robust_y)), float(np.nanmax(robust_y))))
    if bad_centroid.any():
        cols = [c for c in ["cell_id", "cell", "label_id", "region"] if c in gdf.columns]
        display(
            pd.DataFrame({
                "index": gdf.index.astype(str),
                **{c: gdf[c].astype(str).to_numpy() for c in cols},
                "cent_x": raw_cent.x.to_numpy(),
                "cent_y": raw_cent.y.to_numpy(),
                "minx": bounds.minx.to_numpy(),
                "miny": bounds.miny.to_numpy(),
                "maxx": bounds.maxx.to_numpy(),
                "maxy": bounds.maxy.to_numpy(),
                "is_valid": gdf.geometry.is_valid.to_numpy(),
            }).loc[bad_centroid].head(20)
        )
else:
    print(f"No direct shape parquet found at {aligned_shape_path}")


## Feature Preparation

The Spateo basic tutorial recommends PCA across both slices, not independently. `prepare_pair` optionally selects HVGs first, intersects shared genes, and then assigns a joint PCA representation to each slice.

In [ ]:
def _sample_adata(adata: ad.AnnData, max_cells: int | None, seed: int) -> ad.AnnData:
    if max_cells is None or adata.n_obs <= max_cells:
        return adata.copy()
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(adata.n_obs, size=max_cells, replace=False))
    return adata[idx].copy()


def prepare_pair(
    *,
    use_hvg: bool = True,
    n_top_genes: int = 100,
    use_pca: bool = True,
    n_pcs: int = 50,
    max_cells: int | None = None,
    seed: int = 0,
) -> tuple[ad.AnnData, ad.AnnData, dict[str, object]]:
    fixed = prepare_spateo_features(
        fixed_base.copy(),
        normalize_total=10_000.0,
        log1p=True,
        use_hvg=use_hvg,
        n_top_genes=n_top_genes,
    )
    moving = prepare_spateo_features(
        moving_base.copy(),
        normalize_total=10_000.0,
        log1p=True,
        use_hvg=use_hvg,
        n_top_genes=n_top_genes,
    )
    fixed, moving = shared_gene_subset(fixed, moving)
    fixed = _sample_adata(fixed, max_cells=max_cells, seed=seed)
    moving = _sample_adata(moving, max_cells=max_cells, seed=seed + 1)

    rep_kwargs: dict[str, object] = {}
    if use_pca:
        combo = ad.concat(
            {"xenium": fixed, "merscope": moving},
            label="batch",
            index_unique="-",
            merge="same",
        )
        n_comps = min(n_pcs, max(2, fixed.n_vars - 1), max(2, combo.n_obs - 1))
        sc.tl.pca(combo, n_comps=n_comps)
        fixed.obsm["X_pca"] = combo[combo.obs["batch"] == "xenium"].obsm["X_pca"].copy()
        moving.obsm["X_pca"] = combo[combo.obs["batch"] == "merscope"].obsm["X_pca"].copy()
        rep_kwargs = {
            "rep_layer": "X_pca",
            "rep_field": "obsm",
            "dissimilarity": "cos",
        }

    print(
        f"prepared fixed={fixed.shape}, moving={moving.shape}, "
        f"features={fixed.n_vars}, pca={use_pca}, max_cells={max_cells}"
    )
    return fixed, moving, rep_kwargs

## Plotting and Simple Diagnostics

These diagnostics are not a substitute for visual inspection, but they help flag collapse: very high non-rigid displacement, worse nearest-neighbor distances than rigid, or suspiciously compact transformed tissue.

In [ ]:
def _downsample_xy(xy: np.ndarray, n: int = 180_000, seed: int = 0) -> np.ndarray:
    if len(xy) <= n:
        return xy
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(xy), size=n, replace=False)
    return xy[idx]


def _obsm_or(adata: ad.AnnData, key: str, fallback: str = "spatial") -> np.ndarray:
    return np.asarray(adata.obsm[key] if key in adata.obsm else adata.obsm[fallback], dtype=float)


def nearest_neighbor_summary(fixed_xy: np.ndarray, moving_xy: np.ndarray) -> dict[str, float]:
    f_tree = cKDTree(fixed_xy)
    m_tree = cKDTree(moving_xy)
    m_to_f, _ = f_tree.query(moving_xy, k=1)
    f_to_m, _ = m_tree.query(fixed_xy, k=1)
    both = np.r_[m_to_f, f_to_m]
    return {
        "nn_p50": float(np.nanpercentile(both, 50)),
        "nn_p90": float(np.nanpercentile(both, 90)),
        "nn_p99": float(np.nanpercentile(both, 99)),
        "nn_mean": float(np.nanmean(both)),
    }


def displacement_summary(raw_xy: np.ndarray, aligned_xy: np.ndarray) -> dict[str, float]:
    d = np.linalg.norm(aligned_xy - raw_xy, axis=1)
    return {
        "disp_p50": float(np.nanpercentile(d, 50)),
        "disp_p90": float(np.nanpercentile(d, 90)),
        "disp_p99": float(np.nanpercentile(d, 99)),
        "disp_max": float(np.nanmax(d)),
    }


def plot_trial(
    fixed: ad.AnnData,
    moving: ad.AnnData,
    aligned_fixed: ad.AnnData,
    aligned_moving: ad.AnnData,
    *,
    title: str,
    output_path: Path | None = None,
) -> None:
    fixed_raw = np.asarray(fixed.obsm["spatial"], dtype=float)
    moving_raw = np.asarray(moving.obsm["spatial"], dtype=float)
    fixed_common = _obsm_or(aligned_fixed, "align_spatial")
    moving_rigid = _obsm_or(aligned_moving, "align_spatial_rigid", "align_spatial")
    moving_nonrigid = _obsm_or(aligned_moving, "align_spatial_nonrigid", "align_spatial")

    panels = [
        ("Before", fixed_raw, moving_raw),
        ("Rigid", fixed_common, moving_rigid),
        ("Non-rigid", fixed_common, moving_nonrigid),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True, dpi=200)
    for i, (ax, (panel, fxy, mxy)) in enumerate(zip(axes, panels, strict=True)):
        fxy_plot = _downsample_xy(fxy, seed=10 + i)
        mxy_plot = _downsample_xy(mxy, seed=20 + i)
        ax.scatter(fxy_plot[:, 0], fxy_plot[:, 1], s=2, c="#00ff66", alpha=0.12, linewidths=0, rasterized=True, label="Xenium")
        ax.scatter(mxy_plot[:, 0], mxy_plot[:, 1], s=2, c="#ff00cc", alpha=0.12, linewidths=0, rasterized=True, label="MERSCOPE")
        ax.set_title(f"{title}: {panel}")
        ax.set_aspect("equal", adjustable="box")
        ax.invert_yaxis()
        ax.legend(loc="best", markerscale=8)
    if output_path is not None:
        fig.savefig(output_path)
        print(output_path.resolve())
    plt.show()

## Spateo Trial Runner

`batch_size` here corresponds to MerXen's pipeline parameter `alignment_n_sampling`. The trial runner saves only small artifacts: params JSON, coordinate CSVs, overlay PNG, and metric JSON.

In [ ]:
BASE_SPATEO_PARAMS = {
    "mode": "SN-S",
    "allow_flip": True,
    "max_iter": 300,
    "partial_robust_level": 50,
    "beta": 0.1,
    "lambdaVF": 100.0,
    "K": 15,
    "SVI_mode": True,
    "batch_size": 1000,
    "sparse_calculation_mode": True,
    "use_chunk": True,
    "chunk_capacity": 1,
    "verbose": True,
}


def run_trial(
    name: str,
    *,
    feature_cfg: dict[str, object] | None = None,
    spateo_params: dict[str, object] | None = None,
) -> dict[str, object]:
    feature_cfg = feature_cfg or {}
    params = BASE_SPATEO_PARAMS | (spateo_params or {})
    trial_dir = EXPERIMENT_ROOT / name
    trial_dir.mkdir(parents=True, exist_ok=True)

    fixed, moving, rep_kwargs = prepare_pair(**feature_cfg)
    call_params = params | rep_kwargs

    with (trial_dir / "params.json").open("w") as fh:
        json.dump({"feature_cfg": feature_cfg, "spateo_params": params, "rep_kwargs": rep_kwargs}, fh, indent=2)

    t0 = time.time()
    aligned_slices, pis = st.align.morpho_align(
        models=[fixed.copy(), moving.copy()],
        spatial_key="spatial",
        key_added="align_spatial",
        device=DEVICE,
        **call_params,
    )
    elapsed = time.time() - t0
    aligned_fixed, aligned_moving = aligned_slices

    fixed_common = _obsm_or(aligned_fixed, "align_spatial")
    moving_raw = np.asarray(moving.obsm["spatial"], dtype=float)
    moving_rigid = _obsm_or(aligned_moving, "align_spatial_rigid", "align_spatial")
    moving_nonrigid = _obsm_or(aligned_moving, "align_spatial_nonrigid", "align_spatial")

    metrics = {
        "name": name,
        "elapsed_sec": elapsed,
        "n_fixed": int(fixed.n_obs),
        "n_moving": int(moving.n_obs),
        "n_features": int(fixed.n_vars),
        "rigid": nearest_neighbor_summary(fixed_common, moving_rigid) | displacement_summary(moving_raw, moving_rigid),
        "nonrigid": nearest_neighbor_summary(fixed_common, moving_nonrigid) | displacement_summary(moving_raw, moving_nonrigid),
    }
    with (trial_dir / "metrics.json").open("w") as fh:
        json.dump(metrics, fh, indent=2)

    pd.DataFrame({"x": fixed_common[:, 0], "y": fixed_common[:, 1]}).to_csv(trial_dir / "xenium_common.csv", index=False)
    pd.DataFrame({
        "raw_x": moving_raw[:, 0],
        "raw_y": moving_raw[:, 1],
        "rigid_x": moving_rigid[:, 0],
        "rigid_y": moving_rigid[:, 1],
        "nonrigid_x": moving_nonrigid[:, 0],
        "nonrigid_y": moving_nonrigid[:, 1],
    }).to_csv(trial_dir / "merscope_coords.csv", index=False)

    plot_trial(fixed, moving, aligned_fixed, aligned_moving, title=name, output_path=trial_dir / "overlay.png")
    print(json.dumps(metrics, indent=2))
    return {"fixed": fixed, "moving": moving, "aligned_slices": aligned_slices, "pis": pis, "metrics": metrics, "trial_dir": trial_dir}


## First Recommended Trials

Start with subsampled runs. Once a setting is visually stable, remove `max_cells` or increase it for a full-cell no-zarr-rewrite run.

Notes:

- `SN-S` gives the default Spateo behavior where `align_spatial` is rigid. Always inspect `_rigid` and `_nonrigid` separately.
- `SN-N` is worth testing when you truly want a non-rigid output.
- `use_hvg=False` uses the full shared MerXen panel, which may stabilize matching.
- Try `SVI_mode=False` only on subsamples first.

In [ ]:
# Example single trial. Adjust max_cells upward once the settings look sane.
trial = run_trial(
    "partial_conservative_panel_pca_subsample",
    feature_cfg={
        "use_hvg": False,
        "use_pca": True,
        "n_pcs": 50,
        "max_cells": 60_000,
        "seed": 1,
    },
    spateo_params={
        "mode": "SN-S",
        "max_iter": 300,
        "partial_robust_level": 100,
        "beta": 0.01,
        "lambdaVF": 1000.0,
        "K": 15,
        "SVI_mode": True,
    },
)

## Small Parameter Grid

This grid is intentionally small. Add candidates only after inspecting the overlays. Partial-overlap docs suggest testing `partial_robust_level`; non-rigid docs suggest testing `beta`, `lambdaVF`, and `K`; efficiency docs suggest testing SVI carefully.

In [ ]:
TRIAL_SPECS = [
    {
        "name": "rigid_reference_hvg_pca",
        "feature_cfg": {"use_hvg": True, "n_top_genes": 100, "use_pca": True, "max_cells": 60_000, "seed": 2},
        "spateo_params": {"mode": "SN-S", "max_iter": 120, "partial_robust_level": 50, "beta": 0.1, "lambdaVF": 100.0, "K": 15},
    },
    {
        "name": "panel_partial_strict_low_flex",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "max_cells": 60_000, "seed": 3},
        "spateo_params": {"mode": "SN-S", "max_iter": 300, "partial_robust_level": 100, "beta": 0.01, "lambdaVF": 1000.0, "K": 15},
    },
    {
        "name": "panel_partial_moderate_flex",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "max_cells": 60_000, "seed": 4},
        "spateo_params": {"mode": "SN-S", "max_iter": 300, "partial_robust_level": 75, "beta": 0.1, "lambdaVF": 100.0, "K": 30},
    },
    {
        "name": "panel_partial_no_svi_subsample",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "max_cells": 25_000, "seed": 5},
        "spateo_params": {"mode": "SN-S", "max_iter": 300, "partial_robust_level": 100, "beta": 0.01, "lambdaVF": 1000.0, "K": 15, "SVI_mode": False},
    },
    {
        "name": "panel_sn_n_conservative",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "max_cells": 60_000, "seed": 6},
        "spateo_params": {"mode": "SN-N", "max_iter": 300, "partial_robust_level": 100, "beta": 0.01, "lambdaVF": 1000.0, "K": 15},
    },
]

# Uncomment to run the grid.
grid_results = []
for spec in TRIAL_SPECS:
    grid_results.append(run_trial(**spec))

In [ ]:
SECOND_ROUND_SPECS = [
    {
        "name": "strict_delayed_nonrigid_lambda3000",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 60_000, "seed": 11},
        "spateo_params": {
            "mode": "SN-S", "max_iter": 260, "nonrigid_start_iter": 180,
            "partial_robust_level": 100, "beta": 0.005, "lambdaVF": 3000.0,
            "K": 10, "SVI_mode": True, "sparse_top_k": 512,
        },
    },
    {
        "name": "strict_delayed_nonrigid_lambda5000",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 60_000, "seed": 12},
        "spateo_params": {
            "mode": "SN-S", "max_iter": 300, "nonrigid_start_iter": 220,
            "partial_robust_level": 100, "beta": 0.003, "lambdaVF": 5000.0,
            "K": 10, "SVI_mode": True, "sparse_top_k": 512,
        },
    },
    {
        "name": "strict_low_k_high_regularization",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 60_000, "seed": 13},
        "spateo_params": {
            "mode": "SN-S", "max_iter": 300, "nonrigid_start_iter": 200,
            "partial_robust_level": 100, "beta": 0.003, "lambdaVF": 10000.0,
            "K": 8, "SVI_mode": True, "sparse_top_k": 256,
        },
    },
    {
        "name": "strict_fewer_pcs",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 30, "max_cells": 60_000, "seed": 14},
        "spateo_params": {
            "mode": "SN-S", "max_iter": 260, "nonrigid_start_iter": 180,
            "partial_robust_level": 100, "beta": 0.005, "lambdaVF": 3000.0,
            "K": 10, "SVI_mode": True, "sparse_top_k": 512,
        },
    },
    {
        "name": "strict_no_svi_subsample",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 30_000, "seed": 15},
        "spateo_params": {
            "mode": "SN-S", "max_iter": 260, "nonrigid_start_iter": 180,
            "partial_robust_level": 100, "beta": 0.005, "lambdaVF": 3000.0,
            "K": 10, "SVI_mode": False, "sparse_top_k": 512,
        },
    },
]

second_round_results = []
for spec in SECOND_ROUND_SPECS:
    second_round_results.append(run_trial(**spec))


In [ ]:
CONTROL_SPECS = [
    {
        "name": "control_svi_true_25k_seed15",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 15},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 260,
            "partial_robust_level": 100,
            "beta": 0.005,
            "lambdaVF": 3000.0,
            "K": 10,
            "SVI_mode": True,
            "batch_size": 1000,
            "sparse_top_k": 512,
        },
    },
    {
        "name": "control_svi_false_25k_seed15",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 15},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 260,
            "partial_robust_level": 100,
            "beta": 0.005,
            "lambdaVF": 3000.0,
            "K": 10,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
]

control_results = []
for spec in CONTROL_SPECS:
    control_results.append(run_trial(**spec))


In [ ]:
SVI_FALSE_RELAXED_SWEEP = [
    {
        "name": "no_svi_baseline_strict_low_flex",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 21},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 260,
            "nonrigid_start_iter": 160,
            "partial_robust_level": 100,
            "beta": 0.005,
            "lambdaVF": 3000.0,
            "K": 10,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
    {
        "name": "no_svi_relaxed_lambda1000_beta01_k15",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 22},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 280,
            "nonrigid_start_iter": 160,
            "partial_robust_level": 100,
            "beta": 0.01,
            "lambdaVF": 1000.0,
            "K": 15,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
    {
        "name": "no_svi_relaxed_lambda500_beta02_k15",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 23},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 280,
            "nonrigid_start_iter": 140,
            "partial_robust_level": 100,
            "beta": 0.02,
            "lambdaVF": 500.0,
            "K": 15,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
    {
        "name": "no_svi_relaxed_k30_lambda1000",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 24},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 280,
            "nonrigid_start_iter": 160,
            "partial_robust_level": 100,
            "beta": 0.01,
            "lambdaVF": 1000.0,
            "K": 30,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
    {
        "name": "no_svi_more_relaxed_lambda300_beta05_k30",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 25},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 300,
            "nonrigid_start_iter": 120,
            "partial_robust_level": 100,
            "beta": 0.05,
            "lambdaVF": 300.0,
            "K": 30,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
    {
        "name": "no_svi_less_partial_robust_more_flex",
        "feature_cfg": {"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": 25_000, "seed": 26},
        "spateo_params": {
            "mode": "SN-S",
            "max_iter": 280,
            "nonrigid_start_iter": 140,
            "partial_robust_level": 75,
            "beta": 0.02,
            "lambdaVF": 500.0,
            "K": 30,
            "SVI_mode": False,
            "sparse_top_k": 512,
        },
    },
]

svi_false_relaxed_results = []
for spec in SVI_FALSE_RELAXED_SWEEP:
    svi_false_relaxed_results.append(run_trial(**spec))

sweep_names = {spec["name"] for spec in SVI_FALSE_RELAXED_SWEEP}

def collect_metrics(root: Path = EXPERIMENT_ROOT) -> pd.DataFrame:
    rows = []
    for path in sorted(root.glob("*/metrics.json")):
        metrics = json.loads(path.read_text())
        base = {k: v for k, v in metrics.items() if k not in {"rigid", "nonrigid"}}
        for mode in ["rigid", "nonrigid"]:
            row = base | {"alignment_mode": mode} | metrics.get(mode, {})
            rows.append(row)
    return pd.DataFrame(rows)
metrics_df = collect_metrics()
metrics_df[metrics_df["name"].isin(sweep_names)].sort_values(["name", "alignment_mode"])


In [ ]:
def collect_metrics(root: Path = EXPERIMENT_ROOT) -> pd.DataFrame:
    rows = []
    for path in sorted(root.glob("*/metrics.json")):
        metrics = json.loads(path.read_text())
        base = {k: v for k, v in metrics.items() if k not in {"rigid", "nonrigid"}}
        for mode in ["rigid", "nonrigid"]:
            row = base | {"alignment_mode": mode} | metrics.get(mode, {})
            rows.append(row)
    return pd.DataFrame(rows)


metrics_df = collect_metrics()
metrics_df.sort_values(["name", "alignment_mode"]) if len(metrics_df) else metrics_df

In [ ]:
final_trial = run_trial(
    "no_svi_final_strict_low_flex_35k",
    feature_cfg={
        "use_hvg": False,
        "use_pca": True,
        "n_pcs": 50,
        "max_cells": 35_000,
        "seed": 21,
    },
    spateo_params={
        "mode": "SN-S",
        "max_iter": 360,
        "nonrigid_start_iter": 220,
        "partial_robust_level": 100,
        "beta": 0.005,
        "lambdaVF": 3000.0,
        "K": 15,
        "SVI_mode": False,
        "sparse_top_k": 512,
    },
)


In [ ]:
def collect_metrics(root: Path = EXPERIMENT_ROOT) -> pd.DataFrame:
    rows = []
    for path in sorted(root.glob("*/metrics.json")):
        metrics = json.loads(path.read_text())
        base = {k: v for k, v in metrics.items() if k not in {"rigid", "nonrigid"}}
        for mode in ["rigid", "nonrigid"]:
            row = base | {"alignment_mode": mode} | metrics.get(mode, {})
            rows.append(row)
    return pd.DataFrame(rows)

final_metrics_df = collect_metrics()
final_metrics_df[
    final_metrics_df["name"].eq("no_svi_final_strict_low_flex_35k")
].sort_values("alignment_mode")



## Full-Cell No-Rewrite Confirmation

When a subsampled setting looks good, run the same setting with `max_cells=None`. This still writes only overlay/CSV/JSON artifacts, not aligned zarrs.

In [ ]:
# Edit this block after choosing a promising subsample setting.
# full_trial = run_trial(
#     "full_panel_partial_strict_low_flex",
#     feature_cfg={"use_hvg": False, "use_pca": True, "n_pcs": 50, "max_cells": None, "seed": 10},
#     spateo_params={
#         "mode": "SN-S",
#         "max_iter": 300,
#         "partial_robust_level": 100,
#         "beta": 0.01,
#         "lambdaVF": 1000.0,
#         "K": 15,
#         "SVI_mode": True,
#     },
# )

## Convert a Winning Trial Back to MerXen Pipeline Params

For Nextflow, use `alignment_n_sampling` for Spateo's direct `batch_size`. If non-rigid still looks risky, set `alignment_selected_mode=rigid` and keep non-rigid only for inspection.

In [ ]:
winning = {
    "alignment_spateo_mode": "SN-S",
    "alignment_selected_mode": "rigid",
    "alignment_allow_flip": True,
    "alignment_max_iter": 300,
    "alignment_beta": 0.01,
    "alignment_lambda_vf": 1000.0,
    "alignment_k": 15,
    "alignment_partial_robust_level": 100,
    "alignment_svi_mode": True,
    "alignment_n_sampling": 1000,
    "alignment_sparse_calculation_mode": True,
    "alignment_use_chunk": True,
    "alignment_chunk_capacity": 1,
    "alignment_use_hvg": False,
}

for key, value in winning.items():
    print(f"--{key} {str(value).lower() if isinstance(value, bool) else value}")